# Motion-S: Text-to-Sign Motion Generation — Baseline Notebook

This notebook implements a baseline solution for the Signvrse Kaggle competition.
It generates 6 layers of RVQ motion tokens from English/glossified text input.

## Pipeline Overview
1. **Text Encoding** — Encode input text with a CLIP/BERT model
2. **Length Estimation** — Predict sequence length using `length_estimator.pth`
3. **Token Generation** — Generate 6 layers of discrete tokens with a lightweight
   Transformer trained on the provided `train.csv`
4. **Submission** — Format and save `submission.csv`

> **Note:** The `rvq_vae_best.pth` model is used only for *decoding* tokens back to
> motion (optional visualisation). Your task is to generate the tokens, not to run
> the VAE encoder.


## 1. Environment Setup

In [ ]:
# Install required packages (runs silently in Kaggle environment)
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

pip_install('transformers', 'sentencepiece', 'ftfy', 'regex')
print('Packages ready.')

In [ ]:
import os
import math
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import CLIPTokenizer, CLIPTextModel
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
INPUT_DIR        = Path('/kaggle/input/signvrse-motion-generation')  # adjust if needed
TRAIN_CSV        = INPUT_DIR / 'train.csv'
TEST_CSV         = INPUT_DIR / 'test.csv'
VAE_CKPT         = INPUT_DIR / 'rvq_vae_best.pth'
LEN_EST_CKPT     = INPUT_DIR / 'length_estimator.pth'
SUBMISSION_PATH  = Path('submission.csv')

# ── Model hyperparameters ─────────────────────────────────────────────────────
CODEBOOK_SIZE    = 512       # tokens ∈ [0, 511]
NUM_LAYERS       = 6         # base + 5 residuals
MIN_SEQ_LEN      = 40
MAX_SEQ_LEN      = 800
CLIP_MODEL_NAME  = 'openai/clip-vit-base-patch32'
TEXT_EMBED_DIM   = 512       # CLIP text embedding dimension

# Transformer token-generator settings
GEN_D_MODEL      = 256
GEN_NHEAD        = 8
GEN_NUM_LAYERS   = 4
GEN_DIM_FF       = 512
GEN_DROPOUT      = 0.1

# Training
TRAIN_EPOCHS     = 30
BATCH_SIZE       = 32
LR               = 3e-4
GRAD_CLIP        = 1.0

# Generation
TEMPERATURE      = 1.0       # softmax temperature; lower = more confident
TOP_K            = 64        # top-k sampling per step

print('Configuration loaded.')

## 3. Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f'Train samples : {len(train_df)}')
print(f'Test  samples : {len(test_df)}')
print()
print('Train columns :', list(train_df.columns))
print('Test  columns :', list(test_df.columns))
train_df.head(3)

In [ ]:
TOKEN_COLS = ['base_tokens', 'residual_1', 'residual_2',
              'residual_3', 'residual_4', 'residual_5']

def parse_tokens(cell) -> np.ndarray:
    """Convert a space-separated token string to an int32 NumPy array."""
    return np.array(str(cell).strip().split(), dtype=np.int32)

# Parse all token columns in the training set
for col in TOKEN_COLS:
    if col in train_df.columns:
        train_df[col + '_arr'] = train_df[col].apply(parse_tokens)

# Sequence length (all layers should be identical per row)
if 'base_tokens_arr' in train_df.columns:
    train_df['seq_len'] = train_df['base_tokens_arr'].apply(len)
    print('Sequence-length statistics:')
    print(train_df['seq_len'].describe())
else:
    print('Token columns not found in training data — using synthetic lengths.')
    train_df['seq_len'] = np.random.randint(40, 200, size=len(train_df))

## 4. Text Encoder (CLIP)

In [ ]:
print('Loading CLIP text encoder …')
clip_tokenizer  = CLIPTokenizer.from_pretrained(CLIP_MODEL_NAME)
clip_text_model = CLIPTextModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_text_model.eval()

@torch.no_grad()
def encode_texts(texts, batch_size=64):
    """
    Encode a list of strings with CLIP and return a (N, 512) float32 tensor.
    Texts longer than 77 tokens are truncated by the CLIP tokenizer.
    """
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = clip_tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=77,
        ).to(DEVICE)
        outputs = clip_text_model(**inputs)
        # Use the pooled output (sentence embedding)
        embeds = outputs.pooler_output  # (B, 512)
        all_embeds.append(embeds.cpu().float())
    return torch.cat(all_embeds, dim=0)

# Determine text column — prefer gloss when available, fall back to sentence
TEXT_COL = 'gloss' if 'gloss' in train_df.columns else 'sentence'
print(f'Using text column: "{TEXT_COL}"')

# Encode training texts
print('Encoding training texts …')
train_texts = train_df[TEXT_COL].fillna('').tolist()
train_text_embeds = encode_texts(train_texts)   # (N_train, 512)

# Encode test texts
print('Encoding test texts …')
test_text_col = 'gloss' if 'gloss' in test_df.columns else 'sentence'
test_texts  = test_df[test_text_col].fillna('').tolist()
test_text_embeds = encode_texts(test_texts)     # (N_test, 512)

print(f'Train embeds: {train_text_embeds.shape}')
print(f'Test  embeds: {test_text_embeds.shape}')

## 5. Length Estimator

In [ ]:
class LengthEstimatorMLP(nn.Module):
    """MLP that maps a CLIP text embedding to a predicted sequence-length bin."""

    # Bins: each bin covers a 10-token range, 40–800  →  77 bins
    LEN_MIN  = 40
    LEN_MAX  = 800
    BIN_SIZE = 10
    NUM_BINS = (LEN_MAX - LEN_MIN) // BIN_SIZE + 1  # 77

    def __init__(self, in_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128),   nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, self.NUM_BINS),
        )

    def forward(self, x):
        return self.net(x)  # logits (B, NUM_BINS)

    @staticmethod
    def bin_to_length(bin_idx: int) -> int:
        """Convert a bin index back to a sequence length."""
        length = LengthEstimatorMLP.LEN_MIN + bin_idx * LengthEstimatorMLP.BIN_SIZE
        return int(np.clip(length, LengthEstimatorMLP.LEN_MIN, LengthEstimatorMLP.LEN_MAX))

    @staticmethod
    def length_to_bin(length: int) -> int:
        bin_idx = (length - LengthEstimatorMLP.LEN_MIN) // LengthEstimatorMLP.BIN_SIZE
        return int(np.clip(bin_idx, 0, LengthEstimatorMLP.NUM_BINS - 1))


# Try to load pre-trained length estimator
len_estimator = LengthEstimatorMLP(in_dim=TEXT_EMBED_DIM).to(DEVICE)

if LEN_EST_CKPT.exists():
    state = torch.load(LEN_EST_CKPT, map_location=DEVICE)
    len_estimator.load_state_dict(state)
    len_estimator.eval()
    print('Loaded pre-trained length estimator.')
else:
    print('length_estimator.pth not found — will train from scratch.')

    # Build targets from training data
    len_targets = torch.tensor(
        [LengthEstimatorMLP.length_to_bin(l) for l in train_df['seq_len']],
        dtype=torch.long,
    )

    # Quick training
    opt_len = AdamW(len_estimator.parameters(), lr=1e-3)
    ds = torch.utils.data.TensorDataset(train_text_embeds, len_targets)
    dl = DataLoader(ds, batch_size=128, shuffle=True)
    len_estimator.train()
    for epoch in range(20):
        total_loss = 0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            loss = F.cross_entropy(len_estimator(xb), yb)
            opt_len.zero_grad()
            loss.backward()
            opt_len.step()
            total_loss += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch {epoch+1:02d}  loss={total_loss/len(dl):.4f}')
    len_estimator.eval()
    print('Length estimator trained.')


@torch.no_grad()
def predict_lengths(text_embeds: torch.Tensor) -> np.ndarray:
    """Predict sequence lengths for a batch of CLIP embeddings."""
    logits = len_estimator(text_embeds.to(DEVICE))   # (N, NUM_BINS)
    bins   = logits.argmax(dim=-1).cpu().numpy()
    return np.array([LengthEstimatorMLP.bin_to_length(b) for b in bins])

train_pred_lengths = predict_lengths(train_text_embeds)
test_pred_lengths  = predict_lengths(test_text_embeds)

print(f'Predicted test lengths — min={test_pred_lengths.min()}, '
      f'max={test_pred_lengths.max()}, '
      f'mean={test_pred_lengths.mean():.1f}')

## 6. Token Generation Model

We use a **non-autoregressive** (NAR) Transformer that, given a CLIP text embedding
and a target sequence length, predicts all 6 token layers simultaneously.  This is
much faster than an autoregressive model and sufficient for a strong baseline.

In [ ]:
# ── Positional encoding ───────────────────────────────────────────────────────

class SinusoidalPE(nn.Module):
    """Fixed sinusoidal positional encoding (added to token embeddings)."""

    def __init__(self, d_model: int, max_len: int = 1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):            # x: (B, T, D)
        return x + self.pe[:, :x.size(1)]


# ── NAR Token Generator ───────────────────────────────────────────────────────

class NARTokenGenerator(nn.Module):
    """
    Non-Autoregressive Transformer for motion token generation.

    Architecture
    ────────────
    1. Project CLIP text embedding → d_model (context vector)
    2. Create a learnable length query embedding, expanded to target length T
    3. Run T queries through a Transformer decoder, attending to context
    4. Output NUM_LAYERS × CODEBOOK_SIZE logits per position

    During training we supervise all 6 layers jointly with cross-entropy loss.
    """

    def __init__(
        self,
        text_dim    : int = 512,
        d_model     : int = 256,
        nhead       : int = 8,
        num_layers  : int = 4,
        dim_ff      : int = 512,
        dropout     : float = 0.1,
        codebook_sz : int = 512,
        num_rvq_layers: int = 6,
        max_seq_len : int = 800,
    ):
        super().__init__()
        self.d_model        = d_model
        self.codebook_sz    = codebook_sz
        self.num_rvq_layers = num_rvq_layers

        # Project text context
        self.text_proj = nn.Linear(text_dim, d_model)

        # Learnable query embedding (will be tiled to length T)
        self.query_embed = nn.Embedding(max_seq_len, d_model)

        # Positional encoding for queries
        self.pos_enc = SinusoidalPE(d_model, max_len=max_seq_len)

        # Transformer decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,   # Pre-LN for stability
        )
        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer, num_layers=num_layers
        )

        # Output heads — one per RVQ layer
        self.out_heads = nn.ModuleList([
            nn.Linear(d_model, codebook_sz) for _ in range(num_rvq_layers)
        ])

    def forward(
        self,
        text_embed : torch.Tensor,   # (B, text_dim)
        seq_len    : int,
    ):
        """
        Returns logits: (B, seq_len, num_rvq_layers, codebook_sz)
        """
        B = text_embed.size(0)

        # Text context: (B, 1, d_model)
        context = self.text_proj(text_embed).unsqueeze(1)  # (B, 1, d_model)

        # Queries: learnable position embeddings for positions 0..seq_len-1
        pos_idx = torch.arange(seq_len, device=text_embed.device)  # (T,)
        queries = self.query_embed(pos_idx).unsqueeze(0).expand(B, -1, -1)  # (B, T, d_model)
        queries = self.pos_enc(queries)  # add sinusoidal PE

        # Transformer decoder
        out = self.transformer_decoder(queries, context)  # (B, T, d_model)

        # Project to logits for each RVQ layer
        logits = torch.stack(
            [head(out) for head in self.out_heads], dim=2
        )  # (B, T, num_rvq_layers, codebook_sz)

        return logits


model = NARTokenGenerator(
    text_dim     = TEXT_EMBED_DIM,
    d_model      = GEN_D_MODEL,
    nhead        = GEN_NHEAD,
    num_layers   = GEN_NUM_LAYERS,
    dim_ff       = GEN_DIM_FF,
    dropout      = GEN_DROPOUT,
    codebook_sz  = CODEBOOK_SIZE,
    num_rvq_layers = NUM_LAYERS,
    max_seq_len  = MAX_SEQ_LEN,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'NARTokenGenerator: {total_params:,} parameters')

## 7. Dataset and Training

In [ ]:
class MotionTokenDataset(Dataset):
    """
    Dataset that yields (text_embed, token_tensor, seq_len) tuples.

    token_tensor shape: (NUM_LAYERS, seq_len)
    All sequences are padded to a fixed `max_len` for batching.
    """

    def __init__(
        self,
        text_embeds : torch.Tensor,   # (N, text_dim)
        df          : pd.DataFrame,
        token_cols  : list,
        max_len     : int = 256,
        pad_id      : int = 0,
    ):
        self.text_embeds = text_embeds
        self.df          = df.reset_index(drop=True)
        self.token_cols  = token_cols
        self.max_len     = max_len
        self.pad_id      = pad_id

        # Pre-parse tokens
        self.tokens = []
        for _, row in df.iterrows():
            layers = []
            for col in token_cols:
                arr_col = col + '_arr'
                if arr_col in df.columns:
                    layers.append(row[arr_col])
                elif col in df.columns:
                    layers.append(parse_tokens(row[col]))
                else:
                    layers.append(np.zeros(40, dtype=np.int32))
            self.tokens.append(layers)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        embed  = self.text_embeds[idx]      # (text_dim,)
        layers = self.tokens[idx]           # list of 6 arrays
        seq_len = len(layers[0])
        T = min(seq_len, self.max_len)

        # Build (NUM_LAYERS, T) token tensor
        token_t = torch.zeros(NUM_LAYERS, T, dtype=torch.long)
        for l, arr in enumerate(layers):
            trimmed = arr[:T]
            token_t[l, :len(trimmed)] = torch.from_numpy(trimmed.astype(np.int64))

        return embed, token_t, T


def collate_fn(batch):
    """Pad all samples in a batch to the same sequence length."""
    embeds, tokens, lengths = zip(*batch)
    max_T = max(lengths)

    padded_tokens = torch.zeros(len(batch), NUM_LAYERS, max_T, dtype=torch.long)
    for i, (tok, T) in enumerate(zip(tokens, lengths)):
        padded_tokens[i, :, :T] = tok

    return (
        torch.stack(embeds),            # (B, text_dim)
        padded_tokens,                  # (B, NUM_LAYERS, max_T)
        torch.tensor(lengths),          # (B,)
    )


# Only build the dataset if token columns are present
has_tokens = all(c + '_arr' in train_df.columns for c in TOKEN_COLS)

if has_tokens:
    # Use sequences up to 256 tokens to keep GPU memory manageable
    TRAIN_MAX_LEN = 256
    train_dataset = MotionTokenDataset(
        train_text_embeds, train_df, TOKEN_COLS, max_len=TRAIN_MAX_LEN
    )
    train_loader  = DataLoader(
        train_dataset,
        batch_size  = BATCH_SIZE,
        shuffle     = True,
        collate_fn  = collate_fn,
        num_workers = 2,
        pin_memory  = True,
    )
    print(f'Training dataset: {len(train_dataset)} samples')
else:
    print('Token columns not found in training CSV — skipping supervised training.')
    has_tokens = False

In [ ]:
if has_tokens:
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)

    best_loss = float('inf')

    for epoch in range(1, TRAIN_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        num_batches = 0

        for text_emb, tokens, lengths in tqdm(train_loader, desc=f'Epoch {epoch}/{TRAIN_EPOCHS}', leave=False):
            text_emb = text_emb.to(DEVICE)   # (B, text_dim)
            tokens   = tokens.to(DEVICE)     # (B, NUM_LAYERS, T)
            # We use the max length in this batch as the target seq_len
            T = tokens.size(-1)

            logits = model(text_emb, T)      # (B, T, NUM_LAYERS, codebook_sz)

            # Reshape for cross-entropy
            # logits → (B*T*NUM_LAYERS, codebook_sz)
            # tokens → (B*T*NUM_LAYERS,)
            B = text_emb.size(0)
            logits_flat = logits.reshape(B * T * NUM_LAYERS, CODEBOOK_SIZE)
            # tokens: (B, NUM_LAYERS, T) → transpose to (B, T, NUM_LAYERS) → flatten
            target_flat = tokens.permute(0, 2, 1).reshape(B * T * NUM_LAYERS)

            # Build length mask to ignore padding positions
            mask = torch.zeros(B, T, dtype=torch.bool, device=DEVICE)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
            mask = mask.unsqueeze(-1).expand(B, T, NUM_LAYERS).reshape(B * T * NUM_LAYERS)

            loss = F.cross_entropy(
                logits_flat[mask], target_flat[mask], label_smoothing=0.1
            )

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            epoch_loss  += loss.item()
            num_batches += 1

        scheduler.step()
        avg_loss = epoch_loss / max(num_batches, 1)

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), 'best_token_gen.pth')

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:03d} | loss={avg_loss:.4f} | best={best_loss:.4f}')

    # Load the best checkpoint
    model.load_state_dict(torch.load('best_token_gen.pth', map_location=DEVICE))
    print('\nTraining complete. Best model loaded.')
else:
    print('Skipping model training — no token columns found.')

## 8. Inference: Generate Tokens for Test Set

In [ ]:
@torch.no_grad()
def generate_tokens(
    text_embed  : torch.Tensor,   # (1, text_dim)
    seq_len     : int,
    temperature : float = TEMPERATURE,
    top_k       : int   = TOP_K,
) -> np.ndarray:
    """
    Generate 6 layers of motion tokens for a single sample.

    Returns
    -------
    tokens : np.ndarray, shape (NUM_LAYERS, seq_len), dtype int32
    """
    seq_len = int(np.clip(seq_len, MIN_SEQ_LEN, MAX_SEQ_LEN))
    model.eval()

    logits = model(text_embed.to(DEVICE), seq_len)  # (1, T, NUM_LAYERS, codebook_sz)
    logits = logits.squeeze(0)                       # (T, NUM_LAYERS, codebook_sz)

    # Apply temperature
    logits = logits / max(temperature, 1e-8)

    # Top-k filtering
    if top_k > 0:
        top_k = min(top_k, CODEBOOK_SIZE)
        values, _ = torch.topk(logits, top_k, dim=-1)
        threshold = values[..., -1:]
        logits = logits.masked_fill(logits < threshold, float('-inf'))

    probs = F.softmax(logits, dim=-1)               # (T, NUM_LAYERS, codebook_sz)

    # Sample tokens
    T, L, C = probs.shape
    probs_flat = probs.reshape(T * L, C)
    sampled = torch.multinomial(probs_flat, num_samples=1).squeeze(-1)  # (T*L,)
    tokens = sampled.reshape(T, L).permute(1, 0)    # (NUM_LAYERS, T)

    return tokens.cpu().numpy().astype(np.int32)


print('Running inference on test set …')
results = []

for i in tqdm(range(len(test_df))):
    text_emb = test_text_embeds[i].unsqueeze(0)   # (1, 512)
    seq_len  = int(test_pred_lengths[i])

    tokens = generate_tokens(text_emb, seq_len)
    # tokens shape: (NUM_LAYERS, seq_len)

    row = {'id': test_df.iloc[i]['id']}
    col_names = ['base_tokens'] + [f'residual_{j}' for j in range(1, NUM_LAYERS)]
    for layer_idx, col_name in enumerate(col_names):
        row[col_name] = ' '.join(map(str, tokens[layer_idx].tolist()))

    results.append(row)

submission_df = pd.DataFrame(results)
print(f'Generated {len(submission_df)} predictions.')
submission_df.head(3)

## 9. Validate and Save Submission

In [ ]:
def validate_submission(df: pd.DataFrame) -> bool:
    """
    Validate submission DataFrame against competition rules.
    Returns True if all checks pass, raises ValueError otherwise.
    """
    required_cols = ['id', 'base_tokens', 'residual_1', 'residual_2',
                     'residual_3', 'residual_4', 'residual_5']
    token_cols = required_cols[1:]

    # 1. Check all required columns present
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f'Missing column: {col}')

    errors = []
    for idx, row in df.iterrows():
        lengths = []
        for col in token_cols:
            tokens = list(map(int, str(row[col]).strip().split()))
            T = len(tokens)
            lengths.append(T)

            # 2. Token values in [0, 511]
            if any(t < 0 or t > 511 for t in tokens):
                errors.append(f'Row {idx}, {col}: tokens out of [0,511]')

            # 3. Sequence length in [40, 800]
            if T < 40 or T > 800:
                errors.append(f'Row {idx}, {col}: seq_len={T} not in [40,800]')

        # 4. All layers same length
        if len(set(lengths)) != 1:
            errors.append(f'Row {idx}: inconsistent layer lengths {lengths}')

    if errors:
        for e in errors[:10]:
            print(f'  ERROR: {e}')
        raise ValueError(f'{len(errors)} validation error(s) found.')

    print(f'Submission validated: {len(df)} rows, all checks passed ✓')
    return True


validate_submission(submission_df)
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f'Saved to {SUBMISSION_PATH}')

## 10. Quick Summary Statistics

In [ ]:
seq_lens = [len(str(r).split()) for r in submission_df['base_tokens']]
print('Submission sequence-length statistics:')
print(pd.Series(seq_lens).describe().to_string())
print()
print('First 3 rows:')
submission_df.head(3)

## Appendix: Optional Local Evaluation

If you have access to `train.csv` you can approximate the competition score locally
using the provided `evaluation_script.py`.

```bash
python evaluation_script.py \
    --submission submission.csv \
    --ground_truth train.csv
```

Note: The local evaluation uses token-statistics features as a lightweight proxy.
The official leaderboard score uses full VAE-decoded motion features.